# Phase 2B — pandapower network build

Assemble the Phase 1 deliverables into a `pandapower` network at 60 Hz. Buses outside the largest connected component (per the Phase 2A.3 audit) are added but tagged `in_service=False` so the load flow ignores them without losing their topology.

**Cross-voltage `lines` → transformers.** Phase 1 modelled voltage step-down as `lines` whose endpoints sit at different `voltage_kv` — physically wrong (a 138 kV substation does not connect to a 13.8 kV distribution bus by a line). Phase 2B detects those edges and replaces them with `pp.create_transformer_from_parameters` so AC load flow can converge.

**Generators.** With 826 MW of load in service, a single Ormoc slack can't carry all of it through cascading submarine cables. Add realistic local dispatch at the four in-service generator buses (Therma Visayas, Tongonan, Palinpinon 1+2) — starting estimates that the Phase 2D audit can refine.

**Slack:** external grid at Ormoc 350 kV substation (`sub_milagro_substation_104`), `vm_pu=1.02` — the Leyte–Luzon HVDC south terminal, standing in for an import path from Luzon.

**Outputs:**
- `backend/data/processed/pp_network.json` — the assembled network, ready for `pp.runpp`.
- `backend/data/processed/bus_index_map.csv` — `bus_id ↔ pp_index` for joining load-flow output back to canonical IDs.

In [1]:
from pathlib import Path
import pandas as pd
import pandapower as pp
import pandapower.topology as top

PROC = Path('../backend/data/processed')
buses = pd.read_csv(PROC / 'buses.csv')
lines = pd.read_csv(PROC / 'lines.csv')
comp_map = pd.read_csv(PROC / 'bus_component_map.csv')

BIG_COMP = 0
SLACK_BUS_ID = 'sub_milagro_substation_104'  # Ormoc 350 kV
SLACK_VM_PU = 1.02

in_service_mask = comp_map.set_index('bus_id')['component_id'].eq(BIG_COMP)
bus_v = buses.set_index('bus_id')['voltage_kv'].to_dict()
print(f'Buses: {len(buses)}  Lines: {len(lines)}')
print(f'In big component (in_service=True): {int(in_service_mask.sum())} buses')
print(f'Outside big component (in_service=False): {int((~in_service_mask).sum())} buses')
assert SLACK_BUS_ID in set(buses['bus_id']), f'slack {SLACK_BUS_ID} not in buses.csv'
assert in_service_mask[SLACK_BUS_ID], 'slack bus must be in the active component'

Buses: 2959  Lines: 2972
In big component (in_service=True): 1224 buses
Outside big component (in_service=False): 1735 buses


## §1 — Initialise the network

60 Hz (Philippines), not the European 50 Hz default.

In [2]:
net = pp.create_empty_network(f_hz=60, name='visayas_phase2')
print(net)

This pandapower network is empty


## §2 — Create buses

In [3]:
bus_index = {}
for r in buses.itertuples(index=False):
    idx = pp.create_bus(
        net,
        vn_kv=float(r.voltage_kv),
        name=r.bus_id,
        zone=r.province,
        in_service=bool(in_service_mask[r.bus_id]),
        geodata=(float(r.lon), float(r.lat)),
    )
    bus_index[r.bus_id] = idx

assert len(net.bus) == len(buses)
print(f'Created {len(net.bus)} buses ({int(net.bus["in_service"].sum())} in service)')

Created 2959 buses (1224 in service)


## §3 — Split lines: same-voltage stay lines, cross-voltage become transformers

In [4]:
lines['v_from'] = lines['from_bus'].map(bus_v)
lines['v_to'] = lines['to_bus'].map(bus_v)
lines['cross_voltage'] = lines['v_from'] != lines['v_to']

as_lines = lines[~lines['cross_voltage']].copy()
as_trafos = lines[lines['cross_voltage']].copy()

n_self_loop = int((as_lines['from_bus'] == as_lines['to_bus']).sum())
assert n_self_loop == 0, f'unexpected self-loops: {n_self_loop}'

print(f'Same-voltage lines: {len(as_lines)}')
print(f'Cross-voltage rows → transformers: {len(as_trafos)}')
as_trafos['hv'] = as_trafos[['v_from', 'v_to']].max(axis=1)
as_trafos['lv'] = as_trafos[['v_from', 'v_to']].min(axis=1)
print()
print('Transformer voltage pairs (hv, lv):')
print(as_trafos.groupby(['hv', 'lv']).size().sort_values(ascending=False).to_string())

Same-voltage lines: 2423
Cross-voltage rows → transformers: 549

Transformer voltage pairs (hv, lv):
hv     lv   
138.0  13.8     345
69.0   13.8      63
230.0  13.8      62
       138.0     28
138.0  69.0      15
350.0  13.8      14
       138.0     13
       230.0      5
230.0  69.0       2
69.0   60.0       1
230.0  60.0       1


## §3a — Create the same-voltage lines

In [5]:
for r in as_lines.itertuples(index=False):
    line_alive = bool(in_service_mask[r.from_bus] and in_service_mask[r.to_bus])
    pp.create_line_from_parameters(
        net,
        from_bus=bus_index[r.from_bus], to_bus=bus_index[r.to_bus],
        length_km=float(r.length_km),
        r_ohm_per_km=float(r.r_ohm_per_km),
        x_ohm_per_km=float(r.x_ohm_per_km),
        c_nf_per_km=0.0,
        max_i_ka=float(r.max_i_ka),
        name=r.line_id,
        in_service=line_alive,
    )

print(f'Created {len(net.line)} lines ({int(net.line["in_service"].sum())} in service)')

Created 2423 lines (1031 in service)


## §3b — Create the transformers

Sensible defaults for an unknown distribution / transmission transformer: `vk_percent = 12`, `vkr_percent = 0.5`, `pfe_kw = 50`, `i0_percent = 0.1`. `sn_mva` sized to the higher-voltage tier so it does not bottleneck the load flow — Phase 2D can right-size if loadings come out implausible.

| HV side | `sn_mva` |
|---|---:|
| 350 kV | 300 |
| 230 kV | 150 |
| 138 kV |  75 |
| 69 / 60 kV |  30 |

In [6]:
SN_MVA_BY_HV = {350.0: 300.0, 230.0: 150.0, 138.0: 75.0, 69.0: 30.0, 60.0: 30.0}
VK_PERCENT, VKR_PERCENT, PFE_KW, I0_PERCENT = 12.0, 0.5, 50.0, 0.1

for r in as_trafos.itertuples(index=False):
    if r.v_from > r.v_to:
        hv_bus_id, lv_bus_id = r.from_bus, r.to_bus
        hv_kv, lv_kv = r.v_from, r.v_to
    else:
        hv_bus_id, lv_bus_id = r.to_bus, r.from_bus
        hv_kv, lv_kv = r.v_to, r.v_from
    sn_mva = SN_MVA_BY_HV.get(hv_kv)
    assert sn_mva is not None, f'no sn_mva default for hv={hv_kv} kV'
    pp.create_transformer_from_parameters(
        net,
        hv_bus=bus_index[hv_bus_id], lv_bus=bus_index[lv_bus_id],
        sn_mva=sn_mva,
        vn_hv_kv=hv_kv, vn_lv_kv=lv_kv,
        vk_percent=VK_PERCENT, vkr_percent=VKR_PERCENT,
        pfe_kw=PFE_KW, i0_percent=I0_PERCENT,
        name=r.line_id,
        in_service=bool(in_service_mask[hv_bus_id] and in_service_mask[lv_bus_id]),
    )

assert len(net.trafo) == len(as_trafos)
print(f'Created {len(net.trafo)} transformers ({int(net.trafo["in_service"].sum())} in service)')
print(f'Sum check: lines + trafos = {len(net.line) + len(net.trafo)} (lines.csv had {len(lines)})')

Created 549 transformers (257 in service)
Sum check: lines + trafos = 2972 (lines.csv had 2972)


## §4 — Create loads

In [7]:
load_rows = buses[buses['p_mw'].notna() & (buses['p_mw'] > 0)]
for r in load_rows.itertuples(index=False):
    pp.create_load(
        net,
        bus=bus_index[r.bus_id],
        p_mw=float(r.p_mw),
        q_mvar=0.0 if pd.isna(r.q_mvar) else float(r.q_mvar),
        name=f'load_{r.bus_id}',
        in_service=bool(in_service_mask[r.bus_id]),
    )

active_load_mw = float(net.load.loc[net.load['in_service'], 'p_mw'].sum())
inactive_load_mw = float(net.load.loc[~net.load['in_service'], 'p_mw'].sum())
print(f'Created {len(net.load)} loads')
print(f'  in service: {int(net.load["in_service"].sum())} loads, {active_load_mw:.1f} MW')
print(f'  out of svc: {int((~net.load["in_service"]).sum())} loads, {inactive_load_mw:.1f} MW')
print(f'  total:      {active_load_mw + inactive_load_mw:.1f} MW')

Created 2747 loads
  in service: 1128 loads, 820.2 MW
  out of svc: 1619 loads, 1461.8 MW
  total:      2282.0 MW


## §4a — Generator dispatch

Phase 1 identified five `bus_type='generator'` substations. Four sit in the big component; Nabas (Aklan) is in the Panay fragment and stays out of service. Without local injection the slack at Ormoc has to push 826 MW through cascading submarine cables — physically impossible and numerically untraceable. These p_mw values are starting estimates; Phase 2D can refine against real dispatch data.

| Generator | Plant | `p_mw` |
|---|---|---:|
| `v1_05therma` (Cebu) | Therma Visayas | 300 |
| `v1_04tongona` (Leyte) | Tongonan geothermal | 120 |
| `v1_06pgpp1` (Negros Oriental) | Palinpinon unit 1 | 100 |
| `v1_06pgpp2` (Negros Oriental) | Palinpinon unit 2 | 80 |
| **Total local injection** | | **600** |

Remaining ~226 MW + losses imported via the Ormoc HVDC slack.

In [8]:
GEN_DISPATCH_MW = {
    'v1_05therma': 300.0,    # Therma Visayas — Cebu
    'v1_04tongona': 120.0,   # Tongonan geothermal — Leyte
    'v1_06pgpp1': 100.0,     # Palinpinon unit 1 — Negros Oriental
    'v1_06pgpp2': 80.0,      # Palinpinon unit 2 — Negros Oriental
}

for bus_id, p_mw in GEN_DISPATCH_MW.items():
    assert in_service_mask[bus_id], f'{bus_id} not in active component'
    pp.create_gen(net, bus=bus_index[bus_id], p_mw=p_mw, vm_pu=1.0,
                  slack=False, name=f'gen_{bus_id}')

load_mw = float(net.load.loc[net.load['in_service'], 'p_mw'].sum())
gen_mw = float(net.gen['p_mw'].sum())
print(f'Generators: {len(net.gen)}, total dispatch {gen_mw:.0f} MW')
print(f'Expected slack import (load − gen, before losses): {load_mw - gen_mw:.0f} MW')

Generators: 4, total dispatch 600 MW
Expected slack import (load − gen, before losses): 220 MW


## §5 — Slack: external grid at Ormoc 350 kV

In [9]:
slack_idx = bus_index[SLACK_BUS_ID]
pp.create_ext_grid(net, bus=slack_idx, vm_pu=SLACK_VM_PU, va_degree=0.0,
                   name=f'ext_grid_{SLACK_BUS_ID}')
print(f'Slack: {SLACK_BUS_ID}  pp_idx={slack_idx}  vn_kv={net.bus.loc[slack_idx, "vn_kv"]}  vm_pu={SLACK_VM_PU}')

Slack: sub_milagro_substation_104  pp_idx=26  vn_kv=350.0  vm_pu=1.02


## §6 — Topology gate

In [10]:
import networkx as nx
unsupplied = top.unsupplied_buses(net)
mg = top.create_nxgraph(net, respect_switches=True, include_out_of_service=False)
pp_components = list(nx.connected_components(mg))
print(f'Unsupplied in-service buses: {len(unsupplied)}')
print(f'In-service connected components: {len(pp_components)}')
if pp_components:
    print(f'  largest: {max(len(c) for c in pp_components)} buses')

expected = int(in_service_mask.sum())
assert len(unsupplied) == 0, f'{len(unsupplied)} in-service buses are unsupplied — slack unreachable'
assert len(pp_components) == 1, f'expected 1 in-service component, got {len(pp_components)}'
assert max(len(c) for c in pp_components) == expected, 'in-service bus count drift'

Unsupplied in-service buses: 0
In-service connected components: 1
  largest: 1224 buses


## §7 — Save

In [11]:
out_net = PROC / 'pp_network.json'
pp.to_json(net, out_net)
print(f'Wrote {out_net} ({out_net.stat().st_size / 1024:.0f} KB)')

out_map = PROC / 'bus_index_map.csv'
pd.DataFrame({'bus_id': list(bus_index.keys()),
              'pp_index': list(bus_index.values())}).to_csv(out_map, index=False)
print(f'Wrote {out_map} ({len(bus_index)} rows)')

Wrote ../backend/data/processed/pp_network.json (1268 KB)
Wrote ../backend/data/processed/bus_index_map.csv (2959 rows)
